<a href="https://colab.research.google.com/github/Luul1649/CAT-1-ML/blob/main/deep_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install yfinance pandas numpy matplotlib scikit-learn xgboost ta tensorflow streamlit


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 105.7 MB/s eta 0:00:00
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=82561d545a33c7ae3a1cf9a92c8e5b437b3afe8ee21caa2da719ce91ede09662
  Stored in directory: /root/.cache/pip/wheels/5c/a1/5f/c6b85a7d9452057be4ce68a8e45d77ba34234a6d46581777c6
Successfully built ta


In [9]:
import yfinance as yf
import pandas as pd

# Download Apple stock data
df = yf.download("AAPL", start="2010-01-01", end="2026-01-01")

# Flatten MultiIndex columns if they exist. yfinance often returns (Metric, Ticker) MultiIndex.
# This ensures column names like 'Open', 'High', 'Low', 'Close', 'Volume' are directly accessible.
if isinstance(df.columns, pd.MultiIndex):
    # Assumes the metric names (Open, High, Close) are at level 0 of the MultiIndex
    df.columns = df.columns.get_level_values(0)

df = df[['Open','High','Low','Close','Volume']]
df.dropna(inplace=True)

print(df.head())

/tmp/ipython-input-764/2402925050.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download("AAPL", start="2010-01-01", end="2026-01-01")
[*********************100%***********************]  1 of 1 completed

Price           Open      High       Low     Close     Volume
Date                                                         
2010-01-04  6.395006  6.427067  6.363545  6.412385  493729600
2010-01-05  6.430064  6.459728  6.389614  6.423472  601904800
2010-01-06  6.423469  6.448938  6.314703  6.321295  552160000
2010-01-07  6.344665  6.352156  6.263765  6.309608  477131200
2010-01-08  6.301219  6.352157  6.264066  6.351557  447610800


In [27]:
import ta

# Returns
df['Return'] = df['Close'].pct_change()

# Moving averages
df['MA_7'] = df['Close'].rolling(7).mean()
df['MA_30'] = df['Close'].rolling(30).mean()

# Volatility
df['Volatility'] = df['Return'].rolling(7).std()

# RSI
df['RSI'] = ta.momentum.RSIIndicator(df['Close']).rsi()

# MACD
df['MACD'] = ta.trend.MACD(df['Close']).macd()

# Drop NaNs
df.dropna(inplace=True)

In [28]:
df['Target_Close'] = df['Close'].shift(-1)
df['Target_Class'] = (df['Close'].shift(-1) > df['Close']).astype(int)
df.dropna(inplace=True)

In [29]:
train_size = int(len(df)*0.8)

train = df[:train_size]
test = df[train_size:]

features = ['MA_7','MA_30','Volatility','RSI','MACD']

X_train = train[features]
y_train = train['Target_Close']

X_test = test[features]
y_test = test['Target_Close']

In [30]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

model_rf = RandomForestRegressor()
model_rf.fit(X_train, y_train)

pred_rf = model_rf.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred_rf))

MAE: 34.93739564008099


In [17]:
# This cell is now empty as its content has been moved to cell M4tPG-LbLVTG.

In [31]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X_train = train[features]
y_train = train['Target_Class']

X_test = test[features]
y_test = test['Target_Class']

clf = RandomForestClassifier()
clf.fit(X_train, y_train)

pred_class = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred_class))

Accuracy: 0.4766708701134931


In [32]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[['Close']])

X = []
y = []

for i in range(60, len(scaled_data)):
    X.append(scaled_data[i-60:i])
    y.append(scaled_data[i])

X, y = np.array(X), np.array(y)

In [33]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

model_lstm = Sequential()
model_lstm.add(LSTM(50, return_sequences=True, input_shape=(X.shape[1],1)))
model_lstm.add(LSTM(50))
model_lstm.add(Dense(1))

model_lstm.compile(optimizer='adam', loss='mean_squared_error')

model_lstm.fit(X, y, epochs=10, batch_size=32)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
122/122 ━━━━━━━━━━━━━━━━━━━━ 11s 63ms/step - loss: 0.0230
Epoch 2/10
122/122 ━━━━━━━━━━━━━━━━━━━━ 10s 62ms/step - loss: 3.0082e-04
Epoch 3/10
122/122 ━━━━━━━━━━━━━━━━━━━━ 6s 52ms/step - loss: 2.5960e-04
Epoch 4/10
122/122 ━━━━━━━━━━━━━━━━━━━━ 11s 61ms/step - loss: 2.6770e-04
Epoch 5/10
122/122 ━━━━━━━━━━━━━━━━━━━━ 10s 62ms/step - loss: 2.6786e-04
Epoch 6/10
122/122 ━━━━━━━━━━━━━━━━━━━━ 9s 55ms/step - loss: 2.1988e-04
Epoch 7/10
122/122 ━━━━━━━━━━━━━━━━━━━━ 7s 60ms/step - loss: 2.0938e-04
Epoch 8/10
122/122 ━━━━━━━━━━━━━━━━━━━━ 11s 64ms/step - loss: 1.9116e-04
Epoch 9/10
122/122 ━━━━━━━━━━━━━━━━━━━━ 6s 52ms/step - loss: 2.1871e-04
Epoch 10/10
122/122 ━━━━━━━━━━━━━━━━━━━━ 8s 62ms/step - loss: 2.2168e-04
